In [ ]:
"""Week 3 - Course analysis with PySpark.

Loads the cleaned enrollment, progress and course data, joins them, and
produces a course-wise summary plus the top completed and top dropped-out courses.

"""
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

# Fallback when running inside an interactive notebook (where __file__ is undefined)
try:
    ROOT = Path(__file__).resolve().parents[1]
    CLEAN = ROOT / "data" / "clean"
    OUT = ROOT / "week3_pyspark" / "output"
except NameError:
    ROOT = Path("/content")
    CLEAN = ROOT
    OUT = ROOT / "output"

spark = (SparkSession.builder.appName("CourseAnalysis")
         .master("local[*]").config("spark.sql.shuffle.partitions", "4").getOrCreate())
spark.sparkContext.setLogLevel("WARN")


read = lambda f: spark.read.csv(str(CLEAN / f), header=True, inferSchema=True)
enrollments, progress, courses = read("enrollments_clean.csv"), read("progress_clean.csv"), read("courses.csv")


joined = (enrollments
          .join(progress.select("enrollment_id", "completion"), "enrollment_id", "inner")
          .join(courses.select("course_id", "course_name"), "course_id", "inner"))


summary = (joined.groupBy("course_id", "course_name")
           .agg(F.count("*").alias("total_enrolled"),
                F.sum(F.when(F.col("status") == "completed", 1).otherwise(0)).alias("completed"),
                F.sum(F.when(F.col("status") == "dropped", 1).otherwise(0)).alias("dropped"),
                F.round(F.avg("completion"), 2).alias("avg_progress"))
           .withColumn("completion_rate_pct", F.round(100 * F.col("completed") / F.col("total_enrolled"), 1))
           .withColumn("dropout_rate_pct", F.round(100 * F.col("dropped") / F.col("total_enrolled"), 1)))

summary = summary.orderBy(F.desc("total_enrolled"), "course_id")

print("Course summary")
summary.show(truncate=False)

print("Top 5 COMPLETED courses")
top_completed = (summary
                 .orderBy(F.desc("completed"), F.desc("completion_rate_pct"), "course_name")
                 .limit(5))
top_completed.select("course_name", "completed", "completion_rate_pct").show(truncate=False)

print("Top 5 DROPPED-OUT courses")
top_dropped = (summary.filter(F.col("dropped") > 0)
               .orderBy(F.desc("dropped"), F.desc("dropout_rate_pct"), "course_name")
               .limit(5))
top_dropped.select("course_name", "dropped", "dropout_rate_pct").show(truncate=False)


OUT.mkdir(parents=True, exist_ok=True)
summary.toPandas().to_csv(OUT / "course_summary.csv", index=False)
top_completed.toPandas().to_csv(OUT / "top_completed.csv", index=False)
top_dropped.toPandas().to_csv(OUT / "top_dropped.csv", index=False)
spark.stop()

Course summary
+---------+-------------------------+--------------+---------+-------+------------+-------------------+----------------+
|course_id|course_name              |total_enrolled|completed|dropped|avg_progress|completion_rate_pct|dropout_rate_pct|
+---------+-------------------------+--------------+---------+-------+------------+-------------------+----------------+
|C105     |PySpark Fundamentals     |7             |2        |1      |61.43       |28.6               |14.3            |
|C101     |Python for Beginners     |3             |2        |0      |85.0        |66.7               |0.0             |
|C102     |SQL Essentials           |3             |2        |0      |93.33       |66.7               |0.0             |
|C106     |Azure Databricks         |3             |1        |1      |48.33       |33.3               |33.3            |
|C103     |MongoDB Basics           |2             |2        |0      |100.0       |100.0              |0.0             |
|C104     |Data A